In [1]:
!pip install -q google-genai


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
load_dotenv(dotenv_path="env")
API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="hi")
print(f" {response.text.strip()}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


 Hello! How can I help you today?


In [2]:
import json
import os
from google.genai import types

exam_files = ["T1(366).pdf", "T2(366).pdf", "T3(366).pdf"]
output_json = "math366_bank.json"


def process_exam_pdf(pdf_path, output_json="math366_bank.json"):
  print(f"جاري رفع ومعالجة الملف: {pdf_path}...")
  uploaded_file = client.files.upload(file=pdf_path)

  extract_prompt = """
    أنت خبير في مناهج الرياضيات (ريض 366).
    قم باستخراج جميع الأسئلة الواردة في ملف الامتحان المرفق بدقة شديدة (الاختيار من متعدد والأسئلة المقالية).

    المطلوب:
    أرجع النتيجة بصيغة JSON فقط كقائمة من الكائنات (Array of Objects) بالهيكل التالي:
    [
      {
        "id": 1,
        "unit": "اسم الوحدة الدراسية",
        "lesson": "اسم الدرس",
        "type": "mcq أو essay",
        "question": "نص المسألة كاملاً مع كتابة كافة الرموز والمعادلات بصيغة LaTeX ($...$ أو $$...$$)",
        "options": ["الخيار A", "الخيار B", "الخيار C", "الخيار D"],
        "answer": "رمز الإجابة الصحيحة أو الحل النهائي",
        "solution_steps": "خطوات الحل وسلم توزيع الدرجات بالتفصيل"
      }
    ]
    """

  response = client.models.generate_content(
      model="gemini-3.6-flash",
      contents=[uploaded_file, extract_prompt],
      config=types.GenerateContentConfig(response_mime_type="application/json"),
  )

  try:
    new_questions = json.loads(response.text)
  except json.JSONDecodeError:
    print(f"فشل تحليل الاستجابة كـ JSON للملف {pdf_path}")
    return

  all_questions = []
  if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
      try:
        all_questions = json.load(f)
      except json.JSONDecodeError:
        all_questions = []

  start_id = len(all_questions) + 1
  for i, q in enumerate(new_questions):
    q["id"] = start_id + i
    all_questions.append(q)

  with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_questions, f, ensure_ascii=False, indent=2)

  print(
      f"تم بنجاح إضافة {len(new_questions)} سؤالاً من {pdf_path}. العدد الإجمالي"
      f" الآن: {len(all_questions)} سؤالاً."
  )


for file_path in exam_files:
  if os.path.exists(file_path):
    process_exam_pdf(file_path, output_json)
  else:
    print(f"الملف غير موجود في المسار الحالي: {file_path}")

جاري رفع ومعالجة الملف: T1(366).pdf...
فشل تحليل الاستجابة كـ JSON للملف T1(366).pdf
جاري رفع ومعالجة الملف: T2(366).pdf...
تم بنجاح إضافة 14 سؤالاً من T2(366).pdf. العدد الإجمالي الآن: 31 سؤالاً.
جاري رفع ومعالجة الملف: T3(366).pdf...
تم بنجاح إضافة 16 سؤالاً من T3(366).pdf. العدد الإجمالي الآن: 47 سؤالاً.


In [3]:
import json
import os

output_json = "math366_bank.json"

if os.path.exists(output_json):
  with open(output_json, "r", encoding="utf-8") as f:
    bank = json.load(f)

  print(f"إجمالي عدد الأسئلة المخزنة: {len(bank)}")
  print("-" * 40)
  print(f"السؤال الأول: {bank[0]}")
else:
  print("ملف بنك الأسئلة غير موجود بعد!")

إجمالي عدد الأسئلة المخزنة: 47
----------------------------------------
السؤال الأول: {'id': 1, 'unit': 'التكامل غير المحدد وتطبيقاته', 'lesson': 'تكامل الدوال المثلثية', 'type': 'mcq', 'question': 'ما مجموعة الدوال الأصلية للدالة $f(x) = \\csc^2 x$ ؟', 'options': ['F(x) = \\csc x + C', 'F(x) = -\\csc x + C', 'F(x) = \\cot x + C', 'F(x) = -\\cot x + C'], 'answer': 'D', 'solution_steps': 'الدالة الأصلية للدالة $f(x) = \\csc^2 x$ هي $\\int \\csc^2 x \\, dx = -\\cot x + C$.'}


In [13]:
def grade_student_answer(question_obj, student_answer):
    grade_prompt = f"""
    أنت معلم رياضيات لمقرر (ريض 366).
    
    السؤال: {question_obj['question']}
    نموذج الإجابة والخطوات: {question_obj['model_answer']}
    الدرجة الكلية: {question_obj['points']}
    
    إجابة الطالب:
    {student_answer}
    
    المطلوب:
    1. صحح خطوات الطالب بدقة.
    2. حدد الدرجة المستحقة من {question_obj['points']}.
    3. وضّح موضع الخطأ (إن وجد) والخطوة الصحيحة بصيغة LaTeX واضحة.
    """
    
    feedback = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=grade_prompt
    )
    return feedback.text

# تجربة تصحيح حل افتراضي للسؤال
dummy_answer = "تكامل المقدار هو $-\\cot(x) + c$"
evaluation = grade_student_answer(sample_quiz[0], dummy_answer)

display(Markdown("### 📝 تقرير التصحيح والملاحظات:"))
display(Markdown(evaluation))

### 📝 تقرير التصحيح والملاحظات:

أهلاً بك. بصفتي معلم المقرر (ريض 366)، إليك تقييم إجابة الطالب وتصحيحها بدقة:

---

### 1. الدرجة المستحقة:
**0 من 5**

---

### 2. تحديد مواضع الخطأ في إجابة الطالب:

وقع الطالب في خطأين رئيسين:
1. **الخلط بين القوانين المثلثية:** كتب الطالب $-\cot(x)$، وهو تكامل الدالة $\csc^2(x)$ وليس $\sec^2(x)$. تكامل دالة $\sec^2(x)$ هو $\tan(x)$ لأن مشتقة الـ $\tan$ هي $\sec^2$.
2. **إهمال معامل الزاوية (قاعدة السلسلة):** نسي الطالب التعامل مع معامل الزاوية ($k = 3$)؛ حيث يجب بقاء الزاوية كما هي $(3x)$ والضرب في معكوس المعامل $\left(\frac{1}{3}\right)$.

---

### 3. التصحيح والخطوات الصحيحة بصيغة LaTeX:

**القانون المستخدم:**
نعلم أن تكامل دالة القاطع تربيع لزاوية خطية يُعطى بالقاعدة:
$$\int \sec^2(kx) \, dx = \frac{1}{k}\tan(kx) + C$$

**خطوات الحل الصحيح:**
بوضع معامل الزاوية $k = 3$ في القانون:

$$\int \sec^2(3x) \, dx = \frac{1}{3}\tan(3x) + C$$

إذن، مجموعة الدوال الأصلية هي:
$$F(x) = \frac{1}{3}\tan(3x) + C$$